## Cell 1 — Clone or update the repo

Pulls fresh code on every notebook run. `cd` into the repo root so all relative paths (`CMaps/`, `stage3/`, `stage2_refactor/`) resolve.

In [ ]:
import os, subprocess

REPO_URL = "https://github.com/m8wei-coder/ECE-228-project"
REPO_DIR = "/content/ECE-228-project"

if os.path.isdir(REPO_DIR):
    print(f"repo exists at {REPO_DIR}, pulling...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    print(f"cloning {REPO_URL} -> {REPO_DIR} ...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())
!ls -la


## Cell 2 — Install dependencies

Colab usually ships with torch / numpy / pandas / scipy / scikit-learn / joblib / pyyaml. We add `torch_geometric` (the dense modules used by `stage3.models.gnn_modules` don't require `torch_scatter` / `torch_sparse`).

In [ ]:
# Core PyG; DenseGCNConv works without torch_scatter / torch_sparse.
!pip install -q torch_geometric

# stage2_refactor extras that may not be on Colab by default.
!pip install -q pyyaml joblib

# Uncomment if you want WandB logging:
# !pip install -q wandb
print("deps installed")


## Cell 3 — Mount Google Drive

All run outputs (checkpoints, summaries, logs) land under `/content/drive/MyDrive/ece228_stage3/` so a Colab disconnect does not lose results. Graph `.npy` files stay in the repo (`stage3/artifacts/*.npy`).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE_ROOT = "/content/drive/MyDrive/ece228_stage3"
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/runs", exist_ok=True)
print("Drive root:", DRIVE_ROOT)


## Cell 4 — Environment self-check

Confirm torch / torch_geometric / our stage2_refactor + stage3 imports all resolve, and report GPU availability.

In [ ]:
import sys
if "/content/ECE-228-project" not in sys.path:
    sys.path.insert(0, "/content/ECE-228-project")

import torch
print("torch:           ", torch.__version__)
print("cuda available:  ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("  device:        ", torch.cuda.get_device_name(0))

import torch_geometric
print("torch_geometric: ", torch_geometric.__version__)

from stage2_refactor.data.io import read_cmapss_table
from stage2_refactor.training.trainer import fit
from stage2_refactor.training.evaluator import rmse_score
from stage3.models.recurrent_gnn import RecurrentGNNFusion
from stage3.models.gnn_modules import GCN
from stage3.build_graph import build_adjacency
import stage3.train_stage3 as t3
print("stage2_refactor + stage3 imports OK")


## Cell 5 — Build adjacency matrices

Recompute the Pearson / physical / union graphs for all four subsets and write them to `stage3/artifacts/adj_{subset}_{method}.npy`. The repo already ships these files (≤4 KB each, whitelisted in `.gitignore`); this cell exists so a fresh clone can regenerate them and so we can swap the threshold for ablation.

In [ ]:
import subprocess, sys

PEARSON_THRESHOLD = 0.3

result = subprocess.run(
    [sys.executable, "-m", "stage3.build_graph",
     "--threshold", str(PEARSON_THRESHOLD)],
    cwd="/content/ECE-228-project",
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:\n", result.stderr)
    raise RuntimeError("build_graph failed")


## Cell 6 — Hyperparameters (the only knob cell)

Edit this cell only. Setting any value to `None` falls back to the per-subset default baked into `stage3/train_stage3.py:SUBSET_CONFIGS` (which mirrors Stage 2 finalists).

Per-subset defaults:
- FD001 / FD003 → GRU,  h=90, L=2, dropout=0.2, lr=5e-4
- FD002 / FD004 → BiGRU, h=60, L=2, dropout=0.1, lr=5e-4

In [ ]:
SUBSET           = "FD001"     # FD001 / FD002 / FD003 / FD004
GRAPH_METHOD     = "physical"  # physical / pearson / union
USE_GNN          = True         # False = pure recurrent ablation control

# Backbone overrides (None = use per-subset default).
RECURRENT_KIND   = None         # gru / bigru
HIDDEN_SIZE      = None
NUM_LAYERS       = None
DROPOUT          = None
LR               = None
BATCH_SIZE       = None
EPOCHS           = None         # default uses 150 (FD001/2/4) or 250 (FD003)

# GNN branch hyperparams.
GNN_HIDDEN       = 32
GNN_LAYERS       = 2
GNN_KIND         = "gcn"        # gcn (gat is reserved for later ablation)
GNN_DROPOUT      = 0.1
GNN_POOL         = "mean"

# Run id + seed.
SEED             = 42
RUN_ID           = None         # None = f"{subset.lower()}_seed{seed}"


## Cell 7 — Train

Reads variables from Cell 6, calls `stage3.train_stage3.run()` with checkpoints + summary going to Drive. The repo-local `stage3/artifacts/` is still used as the graph source (it holds the `.npy` adjacencies).

In [ ]:
import sys, importlib
import stage3.train_stage3 as t3
importlib.reload(t3)

RUN_ID_EFFECTIVE = RUN_ID or f"{SUBSET.lower()}_seed{SEED}"
OUTPUT_DIR       = f"{DRIVE_ROOT}/runs/{SUBSET}/{RUN_ID_EFFECTIVE}"
CHECKPOINT_DIR   = f"{OUTPUT_DIR}/checkpoints"

argv = [
    "--subset",       SUBSET,
    "--graph-method", GRAPH_METHOD,
    "--seed",         str(SEED),
    "--gnn-hidden",   str(GNN_HIDDEN),
    "--gnn-layers",   str(GNN_LAYERS),
    "--gnn-kind",     GNN_KIND,
    "--gnn-dropout",  str(GNN_DROPOUT),
    "--gnn-pool",     GNN_POOL,
    "--run-id",       RUN_ID_EFFECTIVE,
    "--output-dir",   OUTPUT_DIR,
    "--checkpoint-dir", CHECKPOINT_DIR,
]
argv += ["--use-gnn"] if USE_GNN else ["--no-gnn"]

for flag, val in [
    ("--recurrent-kind", RECURRENT_KIND), ("--hidden-size", HIDDEN_SIZE),
    ("--num-layers", NUM_LAYERS), ("--dropout", DROPOUT),
    ("--learning-rate", LR), ("--batch-size", BATCH_SIZE),
    ("--epochs", EPOCHS),
]:
    if val is not None:
        argv += [flag, str(val)]

print("argv:", argv)
saved_argv = sys.argv
sys.argv = ["train_stage3"] + argv
try:
    args = t3.parse_args()
    summary = t3.run(args)
finally:
    sys.argv = saved_argv

print(f"\n>>> test_rmse={summary['test_rmse']:.4f}  test_score={summary['test_score']:.4f}")


## Cell 8 — Evaluate / inspect the run

Reads `summary.json` + `train_log.csv` from the run directory in Drive and prints a tidy report. Re-run after Cell 7 to inspect that specific run.

In [ ]:
import json
import pandas as pd
from pathlib import Path

run_dir = Path(OUTPUT_DIR)
summary = json.loads((run_dir / "summary.json").read_text())

print("=== run summary ===")
print(f"subset:           {summary['subset']}")
print(f"run_id:           {summary['run_id']}")
print(f"seed:             {summary['seed']}")
print(f"use_gnn:          {summary['use_gnn']}")
print(f"graph_method:     {summary.get('graph_method')}")
print(f"backbone:         {summary['recurrent_kind']} (h={summary['hidden_size']}, L={summary['num_layers']}, d={summary['dropout']})")
print(f"best_epoch / val: {summary['best_epoch']}  (val_rmse={summary['best_metric']:.4f})")
print(f"test_rmse:        {summary['test_rmse']:.4f}")
print(f"test_score:       {summary['test_score']:.4f}")
print(f"train_sequences:  {summary['train_sequences']}")
print(f"val_sequences:    {summary['val_sequences']}")
print(f"parameter_count:  {summary['parameter_count']}")
print(f"train_seconds:    {summary['total_train_seconds']:.1f}")

log_path = run_dir / "train_log.csv"
if log_path.exists():
    log = pd.read_csv(log_path)
    cols = [c for c in ["epoch", "train_loss", "train_rmse", "val_loss", "val_rmse", "val_score"] if c in log.columns]
    print("\n=== last 5 epochs ===")
    print(log[cols].tail(5).to_string(index=False))


## Cell 9 (optional) — Batch: 4 subsets × {GNN on, GNN off} × 3 seeds

24 runs total (4 × 2 × 3). Same 3 seeds as Stage 2 finalists for direct comparability. Each run lands at `DRIVE_ROOT/runs/{SUBSET}/{run_id}/` with `run_id` carrying the seed so no two runs collide.

**Resumable**: at the top of each run we check whether `summary.json` already exists in the target directory. If yes, we read it and skip retraining. So if Colab disconnects mid-batch, just re-run this cell — completed runs are skipped, pending ones continue. Set `SKIP_EXISTING=False` to force-rerun everything.

Final report: per-subset table with GNN vs no-GNN mean ± std over the 3 seeds for both test RMSE and test Score. Full per-run details + aggregates go to `DRIVE_ROOT/batch_summary.json`.

In [ ]:
import json, sys, importlib
from pathlib import Path
from statistics import mean, stdev
import stage3.train_stage3 as t3
importlib.reload(t3)

SEEDS         = [7, 42, 123]                # Stage 2 finalist seeds
BATCH_GRAPH   = "physical"
BATCH_EPOCHS  = {"FD001": 150, "FD002": 150, "FD003": 250, "FD004": 150}
SKIP_EXISTING = True                        # False = force retrain everything

SUBSETS = ["FD001", "FD002", "FD003", "FD004"]

results = []
for subset in SUBSETS:
    for use_gnn in [True, False]:
        for seed in SEEDS:
            tag          = "gnn" if use_gnn else "nognn"
            run_id       = f"{subset.lower()}_{tag}_seed{seed}"
            out_dir      = f"{DRIVE_ROOT}/runs/{subset}/{run_id}"
            ckpt_dir     = f"{out_dir}/checkpoints"
            summary_path = Path(out_dir) / "summary.json"

            if SKIP_EXISTING and summary_path.exists():
                print(f"[skip] {run_id}  (summary.json already exists)")
                s = json.loads(summary_path.read_text())
            else:
                argv = [
                    "--subset", subset, "--graph-method", BATCH_GRAPH,
                    "--seed", str(seed), "--epochs", str(BATCH_EPOCHS[subset]),
                    "--run-id", run_id,
                    "--output-dir", out_dir, "--checkpoint-dir", ckpt_dir,
                ]
                argv += ["--use-gnn"] if use_gnn else ["--no-gnn"]
                print(f"\n=== {subset}  use_gnn={use_gnn}  seed={seed}  run_id={run_id} ===")
                saved_argv = sys.argv
                sys.argv   = ["train_stage3"] + argv
                try:
                    args = t3.parse_args()
                    s    = t3.run(args)
                finally:
                    sys.argv = saved_argv

            results.append({
                "subset": subset, "use_gnn": use_gnn, "seed": seed,
                "test_rmse": s["test_rmse"], "test_score": s["test_score"],
                "best_epoch": s["best_epoch"], "params": s["parameter_count"],
                "train_seconds": s.get("total_train_seconds"),
            })

# ---- aggregate by (subset, use_gnn) over seeds ----
def _mean_std(xs):
    xs = list(xs)
    if len(xs) == 0:
        return (float("nan"), float("nan"))
    if len(xs) == 1:
        return (xs[0], 0.0)
    return (mean(xs), stdev(xs))

agg = []
for subset in SUBSETS:
    for use_gnn in [True, False]:
        rs = [r for r in results if r["subset"] == subset and r["use_gnn"] == use_gnn]
        m_r, s_r = _mean_std([r["test_rmse"]  for r in rs])
        m_s, s_s = _mean_std([r["test_score"] for r in rs])
        agg.append({
            "subset": subset, "use_gnn": use_gnn, "n_seeds": len(rs),
            "seeds": [r["seed"] for r in rs],
            "rmse_mean": m_r, "rmse_std": s_r,
            "score_mean": m_s, "score_std": s_s,
        })

# ---- per-subset comparison table ----
print("\n" + "=" * 78)
print("BATCH SUMMARY  (mean \u00b1 std across seeds {7, 42, 123})")
print("=" * 78)
header = f"{'subset':<8}{'use_gnn':<9}{'test_rmse  (mean \u00b1 std)':>30}{'test_score  (mean \u00b1 std)':>30}"
print(header)
print("-" * len(header))
for subset in SUBSETS:
    for use_gnn in [True, False]:
        a = next(x for x in agg if x["subset"] == subset and x["use_gnn"] == use_gnn)
        cell_r = f"{a['rmse_mean']:7.4f} \u00b1 {a['rmse_std']:6.4f}"
        cell_s = f"{a['score_mean']:9.2f} \u00b1 {a['score_std']:7.2f}"
        print(f"{subset:<8}{str(use_gnn):<9}{cell_r:>30}{cell_s:>30}")
    print()

# ---- save full dump to Drive ----
summary_path = Path(DRIVE_ROOT) / "batch_summary.json"
summary_path.write_text(json.dumps({"runs": results, "aggregate": agg}, indent=2))
print(f"Saved: {summary_path}")
print(f"Total runs: {len(results)} / {len(SUBSETS) * 2 * len(SEEDS)}")


## Cell 10 (optional) — Graph method ablation: pearson + union (GNN-only)

Adds the **pearson** and **union** graphs to compare against the **physical** graph already covered by Cell 9. No new runs for `use_gnn=False` here — the pure-recurrent control is graph-agnostic and was already produced in Cell 9.

Total new runs: 2 graphs × 4 subsets × 3 seeds = **24**.

`run_id` carries the graph suffix (e.g. `fd001_gnn_seed7_pearson`) so it cannot collide with Cell 9 outputs (which have no suffix). `SKIP_EXISTING=True` lets a Colab disconnect resume mid-batch.

Final report: per-subset table laying out **gnn-physical** (read from Cell 9's `batch_summary.json`) **+ gnn-pearson + gnn-union + no-gnn baseline** side by side. New results land at `DRIVE_ROOT/batch_summary_graphs.json` (Cell 9's `batch_summary.json` is left untouched).

In [ ]:
import json, sys, importlib
from pathlib import Path
from statistics import mean, stdev
import stage3.train_stage3 as t3
importlib.reload(t3)

SEEDS         = [7, 42, 123]
GRAPH_METHODS = ["pearson", "union"]    # physical already in Cell 9
BATCH_EPOCHS  = {"FD001": 150, "FD002": 150, "FD003": 250, "FD004": 150}
SKIP_EXISTING = True

SUBSETS = ["FD001", "FD002", "FD003", "FD004"]

results = []
for graph_method in GRAPH_METHODS:
    for subset in SUBSETS:
        for seed in SEEDS:
            run_id       = f"{subset.lower()}_gnn_seed{seed}_{graph_method}"
            out_dir      = f"{DRIVE_ROOT}/runs/{subset}/{run_id}"
            ckpt_dir     = f"{out_dir}/checkpoints"
            summary_path = Path(out_dir) / "summary.json"

            if SKIP_EXISTING and summary_path.exists():
                print(f"[skip] {run_id}  (summary.json already exists)")
                s = json.loads(summary_path.read_text())
            else:
                argv = [
                    "--subset", subset, "--graph-method", graph_method,
                    "--seed", str(seed), "--epochs", str(BATCH_EPOCHS[subset]),
                    "--run-id", run_id,
                    "--output-dir", out_dir, "--checkpoint-dir", ckpt_dir,
                    "--use-gnn",
                ]
                print(f"\n=== {subset}  graph={graph_method}  seed={seed}  run_id={run_id} ===")
                saved_argv = sys.argv
                sys.argv   = ["train_stage3"] + argv
                try:
                    args = t3.parse_args()
                    s    = t3.run(args)
                finally:
                    sys.argv = saved_argv

            results.append({
                "subset": subset, "graph_method": graph_method,
                "use_gnn": True, "seed": seed,
                "test_rmse": s["test_rmse"], "test_score": s["test_score"],
                "best_epoch": s["best_epoch"], "params": s["parameter_count"],
                "train_seconds": s.get("total_train_seconds"),
            })

# ---- load Cell 9 results (physical + no-GNN baseline) for side-by-side comparison ----
physical_path       = Path(DRIVE_ROOT) / "batch_summary.json"
physical_aggregates = []
if physical_path.exists():
    physical_aggregates = json.loads(physical_path.read_text()).get("aggregate", [])
else:
    print(f"WARNING: {physical_path} not found; cannot compare against Cell 9 physical / no-gnn.")

def _mean_std(xs):
    xs = list(xs)
    if len(xs) == 0:
        return (float("nan"), float("nan"))
    if len(xs) == 1:
        return (xs[0], 0.0)
    return (mean(xs), stdev(xs))

agg = []
for subset in SUBSETS:
    for graph_method in GRAPH_METHODS:
        rs = [r for r in results if r["subset"] == subset and r["graph_method"] == graph_method]
        m_r, s_r = _mean_std([r["test_rmse"]  for r in rs])
        m_s, s_s = _mean_std([r["test_score"] for r in rs])
        agg.append({
            "subset": subset, "graph_method": graph_method, "n_seeds": len(rs),
            "seeds": [r["seed"] for r in rs],
            "rmse_mean": m_r, "rmse_std": s_r,
            "score_mean": m_s, "score_std": s_s,
        })

def _lookup_cell9(subset, want_use_gnn):
    for a in physical_aggregates:
        if a["subset"] == subset and a["use_gnn"] == want_use_gnn:
            return a
    return None

# ---- side-by-side table ----
print("\n" + "=" * 86)
print("GRAPH METHOD ABLATION  (mean \u00b1 std across seeds {7, 42, 123})")
print("=" * 86)
header = f"{'subset':<8}{'variant':<16}{'test_rmse  (mean \u00b1 std)':>30}{'test_score  (mean \u00b1 std)':>30}"
print(header)
print("-" * len(header))
for subset in SUBSETS:
    # gnn-physical from Cell 9
    p = _lookup_cell9(subset, True)
    if p is not None:
        cell_r = f"{p['rmse_mean']:7.4f} \u00b1 {p['rmse_std']:6.4f}"
        cell_s = f"{p['score_mean']:9.2f} \u00b1 {p['score_std']:7.2f}"
        print(f"{subset:<8}{'gnn (physical)':<16}{cell_r:>30}{cell_s:>30}")
    else:
        print(f"{subset:<8}{'gnn (physical)':<16}{'(not found)':>30}{'(not found)':>30}")
    # gnn-pearson + gnn-union from this cell
    for graph_method in GRAPH_METHODS:
        a = next((x for x in agg if x["subset"] == subset and x["graph_method"] == graph_method), None)
        label = f"gnn ({graph_method})"
        if a is not None and a["n_seeds"] > 0:
            cell_r = f"{a['rmse_mean']:7.4f} \u00b1 {a['rmse_std']:6.4f}"
            cell_s = f"{a['score_mean']:9.2f} \u00b1 {a['score_std']:7.2f}"
        else:
            cell_r, cell_s = "(no runs)", "(no runs)"
        print(f"{subset:<8}{label:<16}{cell_r:>30}{cell_s:>30}")
    # no-gnn baseline from Cell 9
    n = _lookup_cell9(subset, False)
    if n is not None:
        cell_r = f"{n['rmse_mean']:7.4f} \u00b1 {n['rmse_std']:6.4f}"
        cell_s = f"{n['score_mean']:9.2f} \u00b1 {n['score_std']:7.2f}"
        print(f"{subset:<8}{'no-gnn':<16}{cell_r:>30}{cell_s:>30}")
    else:
        print(f"{subset:<8}{'no-gnn':<16}{'(not found)':>30}{'(not found)':>30}")
    print()

out_path = Path(DRIVE_ROOT) / "batch_summary_graphs.json"
out_path.write_text(json.dumps({
    "runs": results,
    "aggregate": agg,
    "physical_from_cell9": physical_aggregates,
}, indent=2))
print(f"Saved: {out_path}")
print(f"Total new runs: {len(results)} / {len(GRAPH_METHODS) * len(SUBSETS) * len(SEEDS)}")
